#### Import Library

In [1]:
import pandas as pd
import numpy as np
import re
import string
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Install resource NLTK jika belum ada
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acern\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acern\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acern\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Pada tahap ini, sistem menyiapkan seluruh library dan resource yang dibutuhkan untuk pemrosesan teks. Data dikelola menggunakan Pandas dan NumPy, kemudian teks dibersihkan dengan modul `re` dan `string`. Selanjutnya, NLTK digunakan untuk melakukan tokenisasi, menghapus stopword, dan stemming. Setelah itu, teks diubah menjadi representasi numerik menggunakan TF-IDF dan tingkat kemiripan antar teks dihitung dengan Cosine Similarity. Terakhir, resource NLTK yang diperlukan diunduh agar seluruh proses dapat berjalan dengan baik.


#### Load Clean Dataset

In [2]:
feature_all_job_df = pd.read_csv('../../data/processed/cleaned_all_job.csv')

Pada tahap ini, sistem membaca dan memuat dataset `cleaned_all_job.csv` ke dalam DataFrame menggunakan Pandas. Data yang telah dibersihkan tersebut kemudian disimpan dalam variabel `feature_all_job_df` untuk digunakan pada proses analisis dan rekomendasi selanjutnya.

#### Text Preprocessing

In [3]:
feature_all_job_df['combined_features'] = (
    feature_all_job_df['category'].fillna('') + ' ' +
    feature_all_job_df['job_title'].fillna('') + ' ' +
    feature_all_job_df['job_description'].fillna('') + ' ' +
    feature_all_job_df['job_skill_set'].fillna('')
)

Pada tahap ini, sistem menggabungkan informasi dari kolom kategori, judul pekerjaan, deskripsi pekerjaan, dan keterampilan yang dibutuhkan ke dalam satu kolom bernama `combined_features`. Penggabungan ini dilakukan untuk membentuk representasi teks yang lebih lengkap sehingga dapat digunakan dalam proses ekstraksi fitur dan perhitungan kemiripan pekerjaan.


In [4]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):

    # lowercase
    text = text.lower()

    # hapus angka
    text = re.sub(r'\d+', '', text)

    # hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenizing
    tokens = word_tokenize(text)

    # hapus stopwords + stemming
    tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words
    ]

    return ' '.join(tokens)

feature_all_job_df['processed_text'] = feature_all_job_df['combined_features'].apply(preprocess_text)

Pada tahap ini, sistem melakukan preprocessing teks dengan mengubah seluruh huruf menjadi huruf kecil, menghapus angka dan tanda baca, memecah teks menjadi token kata, menghapus stopword bahasa Inggris, serta melakukan stemming untuk mengubah kata ke bentuk dasarnya. Hasil preprocessing kemudian disimpan pada kolom `processed_text` sehingga data menjadi lebih bersih dan siap digunakan dalam proses ekstraksi fitur serta perhitungan kemiripan.


#### TF-IDF Vectorization

In [5]:
tfidf = TfidfVectorizer(max_features=5000)

tfidf_matrix = tfidf.fit_transform(feature_all_job_df['processed_text'])

print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)


TF-IDF Matrix Shape:
(1167, 5000)


Pada tahap ini, sistem mengubah teks yang telah diproses menjadi representasi numerik menggunakan metode TF-IDF dengan batas maksimum 5.000 fitur kata. Proses ini menghasilkan matriks TF-IDF yang merepresentasikan tingkat kepentingan setiap kata pada masing-masing data pekerjaan. Selanjutnya, ukuran matriks ditampilkan untuk melihat jumlah data dan fitur yang berhasil terbentuk.


#### Hitung Cosine Similarity

In [6]:
cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Shape:")
print(cosine_sim.shape)

Cosine Similarity Shape:
(1167, 1167)


Pada tahap ini, sistem menghitung tingkat kemiripan antar pekerjaan menggunakan metode Cosine Similarity berdasarkan matriks TF-IDF yang telah dibuat. Hasil perhitungan berupa matriks kemiripan yang menunjukkan seberapa mirip setiap pekerjaan dengan pekerjaan lainnya. Selanjutnya, ukuran matriks ditampilkan untuk memastikan proses perhitungan berhasil dilakukan.


#### Export Feature Matrix


In [22]:
joblib.dump(
    tfidf_matrix,
    '../../data/processed/tfidf_matrix.pkl'
)

joblib.dump(
    cosine_sim,
    '../../data/processed/cosine_similarity.pkl'
)

joblib.dump(
    tfidf,
    '../../data/processed/tfidf_vectorizer.pkl'
)

['../../data/processed/tfidf_vectorizer.pkl']

Pada tahap ini, sistem menyimpan matriks TF-IDF, matriks Cosine Similarity, dan model TF-IDF Vectorizer ke dalam file berformat `.pkl` menggunakan Joblib. Penyimpanan ini dilakukan agar hasil pemrosesan dapat digunakan kembali pada tahap rekomendasi tanpa perlu melakukan perhitungan ulang, sehingga proses menjadi lebih cepat dan efisien.
